In [ ]:
import cv2
from ultralytics import YOLO

def detect_objects_from_usb_camera(model_path='best-111.pt', camera_index=0):
    """
    Perform real-time object detection using YOLOv8 from a USB camera.

    Args:
        model_path (str): Path to the YOLOv8 model to use (e.g., 'yolov8n.pt', 'yolov8m.pt').
                          If not specified, 'yolov8n.pt' (Nano model) will be downloaded automatically.
        camera_index (int): Index of the USB camera to use.
                            Typically, the default camera is 0.
                            If you have multiple cameras, try 1, 2, etc.
    """

    # Load the YOLOv8 model
    # The specified model will be downloaded automatically the first time it's run
    model = YOLO(model_path)

    # Initialize the USB camera
    cap = cv2.VideoCapture(camera_index)

    # Check if the camera was opened successfully
    if not cap.isOpened():
        print(f"Error: Could not open camera {camera_index}. Please check the camera index.")
        return

    print("Starting real-time object detection. Press 'q' to quit.")

    while True:
        # Read a frame from the camera
        ret, frame = cap.read()

        # Check if the frame was read successfully
        if not ret:
            print("Error: Could not read frame. Exiting.")
            break

        # Perform object detection with YOLOv8
        # Using stream=True returns results as a generator, which is memory-efficient
        # for processing a large number of frames.
        results = model(frame, stream=True)

        # Draw detection results on the frame
        for r in results:
            # Get bounding box, class, and confidence for detected objects
            # r.boxes contains the detection results
            for box in r.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0]) # Bounding box coordinates
                confidence = float(box.conf[0])       # Confidence score
                class_id = int(box.cls[0])            # Class ID
                class_name = model.names[class_id]    # Class name

                # Draw the bounding box
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2) # Green rectangle

                # Draw the text (class name and confidence)
                label = f"{class_name}: {confidence:.2f}"
                cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        # Display the result
        cv2.imshow("YOLOv8 USB Camera Detection", frame)

        # Exit the loop if 'q' key is pressed
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Release resources
    cap.release()
    cv2.destroyAllWindows()
    print("Detection stopped.")

if __name__ == "__main__":
    # Use YOLOv8n model with the default USB camera (index 0)
    # If you want to use a different model (e.g., 'yolov8m.pt'), specify model_path='yolov8m.pt'
    # If you have multiple cameras, try changing camera_index=1, etc.
    detect_objects_from_usb_camera(model_path='yolov8n.pt', camera_index=0)